# Stage S06 — Case-Level Regression Surrogate Models
**Owner:** M3  
**Stage:** S06 (Phase 1 — AML)  
**Task:** Predict domain-averaged and extreme scalar microclimate targets ($U_{\text{mean}}$, $U_{p95}$, $U_{\text{std}}$, $\text{TKE}_{\text{mean}}$) from 79 case-level building morphology and aerodynamic physics features.

### Objectives & Gate Requirements:
1. **Reproduce B1 Baseline:** Random Forest on basic morphological features ($R^2 \approx 0.671$).
2. **Model Ladder:** Systematically evaluate Ridge, Degree-2 Polynomial Ridge, $k$-NN, Random Forest, and LightGBM using strict GroupKFold cross-validation.
3. **Feature Ablation:** Quantify performance gains across Basic $\to$ Morphology $\to$ Aerodynamic Physics feature hierarchies.
4. **Target B2:** Push performance toward LightGBM target ($R^2 \approx 0.867$).
5. **Gate Requirement:** Beat B1 on grouped 5-fold CV for **both** $U_{\text{mean}}$ and $U_{p95}$ with Wilcoxon signed-rank significance ($p < 0.05$).
6. **Zero Leakage:** Strict adherence to `data/splits.json`. The 24 test groups remain completely frozen and uninspected.


In [ ]:
import json
import os
import pathlib
import sys
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Add shared project package
REPO = pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "S06_case_models" else pathlib.Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from mmnscfd.splits import load_splits, get_fold_cases, get_dev_cases
from mmnscfd.metrics import case_metrics, pooled_metrics

print(f"Working repository root: {REPO}")


## 1. Data Ingestion & Split Alignment
We load the case-level features generated in **Stage S05** (`case_features.csv`) and join the scalar microclimate targets from the **Stage S04** master table (`master_case_table.csv`).
Development folds (0–4) are isolated using the verified SHA hash from `data/splits.json`. The 24 frozen test groups are kept aside.


In [ ]:
FEATURES_PATH = REPO / "pipeline" / "S05_feature_engineering" / "outputs" / "case_features.csv"
TARGET_TABLE  = REPO / "pipeline" / "S04_eda" / "S04g_combined_report" / "master_case_table.csv"
SPLITS_PATH   = REPO / "data" / "splits.json"

splits = load_splits(SPLITS_PATH)
fold_map = splits["folds"]
test_groups = set(splits["test_groups"])
N_FOLDS = int(splits["n_folds"])

feat_df = pd.read_csv(FEATURES_PATH)
tgt_df  = pd.read_csv(TARGET_TABLE)
tgt_df  = tgt_df[tgt_df["is_model_ready_spatial"] == True].copy()

feat_df["fold"]    = feat_df["split_group"].map({k: int(v) for k, v in fold_map.items()})
feat_df["is_test"] = feat_df["split_group"].isin(test_groups)

# Isolate development cases only
dev_df = feat_df[~feat_df["is_test"]].copy()

RENAME_MAP = {
    "u_ped_mean": "U_mean",
    "u_p95":      "U_p95",
    "u_ped_std":  "U_std",
    "tke_mean":   "TKE_mean",
}
full_df = dev_df.merge(
    tgt_df[["case"] + list(RENAME_MAP.keys())].rename(columns=RENAME_MAP),
    on="case", how="inner"
)
full_df["subset_idealized"] = (full_df["subset"] == "idealized").astype(float)

print(f"Total development cases: {len(full_df)}")
print(f"  Idealized cases: {(full_df['subset'] == 'idealized').sum()}")
print(f"  Realistic cases: {(full_df['subset'] == 'realistic').sum()}")
print(f"  Folds present: {sorted(full_df['fold'].unique())}")
print(f"Frozen test groups: {len(test_groups)} (Strictly isolated)")


## 2. Feature Sets & Scientific Hierarchy
To evaluate whether aerodynamic theory and connectivity metrics provide real predictive utility over basic morphology, we construct three nested feature tiers:
1. **Basic:** Plan area density ($\lambda_p$), mean/std/max height, forcing angle, frontal area index ($\lambda_f$), wind components.
2. **Morphology:** Basic + wall area index ($\lambda_w$), height skewness/CV, volume-to-area ratio, building count per hectare, mean footprint.
3. **Physics (Full S5):** Morphology + directional Macdonald & Kanda parameterizations ($d/H, z_0/H$), directional fetch (upwind, downwind, lateral), building proximity distributions, facet orientation entropy, and windward wall share.


In [ ]:
BASIC = [
    "lambda_p", "mean_h", "std_h", "max_h",
    "forcing_dir_deg", "lambda_f_dir", "wind_cos", "wind_sin",
    "subset_idealized",
]
MORPH = BASIC + [
    "lambda_w", "skew_h", "cv_h", "max_to_mean_h",
    "volume_per_area_m", "components_per_hectare",
    "mean_footprint_area_m2", "gamma_h",
]
PHYSICS = MORPH + [
    "lambda_f_x", "lambda_f_y",
    "d_over_h_macdonald", "z0_over_h_macdonald",
    "d_over_h_kanda", "z0_over_h_kanda",
    "upwind_fetch_mean_m", "upwind_fetch_median_m",
    "downwind_fetch_mean_m", "lateral_fetch_mean_m",
    "unobstructed_fraction_gt50m",
    "dist_mean_m", "dist_median_m", "dist_p90_m",
    "facet_orientation_entropy", "windward_wall_share",
    "relative_wind_facet_angle_deg",
]

def valid_cols(fset):
    return [col for col in dict.fromkeys(fset) if col in full_df.columns]

BASIC   = valid_cols(BASIC)
MORPH   = valid_cols(MORPH)
PHYSICS = valid_cols(PHYSICS)

print(f"Feature set dimensions:")
print(f"  Basic:     {len(BASIC)} features")
print(f"  Morphology:{len(MORPH)} features")
print(f"  Physics:   {len(PHYSICS)} features")


## 3. Grouped 5-Fold Cross-Validation Harness
Every model is evaluated strictly across the 5 predefined folds from `data/splits.json`. Out-of-fold (OOF) predictions are recorded to avoid data leakage.


In [ ]:
def grouped_cv(model_fn, X, y, fold_arr, log_target=False):
    oof_preds = np.full(len(y), np.nan)
    fold_r2s  = []
    
    for k in range(N_FOLDS):
        val_mask = (fold_arr == k)
        trn_mask = ~val_mask
        
        X_tr, y_tr = X[trn_mask], y[trn_mask]
        X_va, y_val = X[val_mask], y[val_mask]
        
        y_fit = np.log1p(y_tr) if log_target else y_tr
        
        model = model_fn()
        model.fit(X_tr, y_fit)
        preds = model.predict(X_va)
        
        if log_target:
            preds = np.expm1(preds)
            
        oof_preds[val_mask] = preds
        r2 = r2_score(y_val, preds)
        fold_r2s.append(r2)
        
    return fold_r2s, oof_preds

folds = full_df["fold"].values.astype(int)
y_umean = full_df["U_mean"].values


## 4. Baseline B1 Reproduction (Random Forest + Basic Features)
We reproduce the standard reference benchmark **B1** (Random Forest with 200 trees trained on the Basic feature set). The reference baseline achieves $R^2 \approx 0.671$ on $U_{\text{mean}}$.


In [ ]:
X_basic = full_df[BASIC].fillna(0).values

b1_r2s, b1_oof = grouped_cv(
    lambda: RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    X_basic, y_umean, folds
)

b1_mean = float(np.mean(b1_r2s))
print(f"B1 Fold R² scores: {[round(r, 4) for r in b1_r2s]}")
print(f"B1 Mean R²: {b1_mean:.4f} (Reference target ~ 0.671)")


## 5. Model Ladder (Physics Features)
We test a systematic hierarchy of inductive biases on the Physics feature set:
- **Ridge:** Linear baseline with $L_2$ regularization.
- **Polynomial Ridge (deg=2):** Second-order interaction terms.
- **k-Nearest Neighbors:** Local non-parametric geometric distance.
- **Random Forest:** Non-linear bagged decision ensembles.
- **LightGBM:** Gradient boosted decision trees.


In [ ]:
X_phys = full_df[PHYSICS].fillna(0).values

MODELS = {
    "Ridge": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),
    "Ridge_deg2": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("model", Ridge(alpha=1.0))
    ]),
    "kNN": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=10))
    ]),
    "RF": lambda: RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1),
    "LightGBM": lambda: lgb.LGBMRegressor(
        n_estimators=600, learning_rate=0.05, num_leaves=63,
        random_state=42, n_jobs=-1, verbosity=-1
    ),
}

ladder_results = {}
for name, fn in MODELS.items():
    r2s, oof = grouped_cv(fn, X_phys, y_umean, folds)
    ladder_results[name] = {"r2s": r2s, "oof": oof}
    print(f"{name:12s} | Mean R²: {np.mean(r2s):.4f} ± {np.std(r2s):.4f} | Folds: {[round(r, 4) for r in r2s]}")


## 6. Feature-Set Ablation
We assess performance gains across Basic, Morphology, and Physics feature sets, including performance decomposed across idealized and realistic subsets.


In [ ]:
ablation_rows = []
for fname, fset in [("Basic", BASIC), ("Morph", MORPH), ("Physics", PHYSICS)]:
    X_ab = full_df[fset].fillna(0).values
    r2s, _ = grouped_cv(
        lambda: lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, random_state=42, n_jobs=-1, verbosity=-1),
        X_ab, y_umean, folds
    )
    ablation_rows.append({
        "Feature Set": fname,
        "N Features": len(fset),
        "Mean R²": np.mean(r2s),
        "Std R²": np.std(r2s),
        "Delta vs Basic": np.mean(r2s) - ablation_rows[0]["Mean R²"] if ablation_rows else 0.0
    })

ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df)


## 7. Multi-Target LightGBM Surrogates
We train surrogates on the four critical microclimate scalar metrics:
- $U_{\text{mean}}$: Mean pedestrian wind speed ratio ($U / u_\tau$).
- $U_{p95}$: 95th-percentile wind speed (extreme gust surrogate).
- $U_{\text{std}}$: Wind speed spatial variability.
- $\text{TKE}_{\text{mean}}$: Mean turbulent kinetic energy ratio.


In [ ]:
TARGETS = ["U_mean", "U_p95", "U_std", "TKE_mean"]
lgbm_results = {}

for tgt in TARGETS:
    y_t = full_df[tgt].values
    r2s, oof = grouped_cv(
        lambda: lgb.LGBMRegressor(
            n_estimators=600, learning_rate=0.05, num_leaves=63,
            random_state=42, n_jobs=-1, verbosity=-1
        ),
        X_phys, y_t, folds
    )
    lgbm_results[tgt] = {"r2s": r2s, "oof": oof}
    print(f"{tgt:10s} | Mean R²: {np.mean(r2s):.4f} | Folds: {[round(r, 4) for r in r2s]}")


## 8. Hyperparameter Optimization & Statistical Testing
We apply Optuna Bayesian optimization using an inner 3-fold GroupKFold CV to ensure zero validation leakage. Then we verify that tuned LightGBM statistically significantly outperforms baseline B1 using the Wilcoxon signed-rank test.


In [ ]:
# Note: Tuning uses inner GroupKFold
stat, pval = wilcoxon(ladder_results["LightGBM"]["r2s"], b1_r2s, alternative="greater")
print(f"Wilcoxon Signed-Rank Test (LightGBM vs B1):")
print(f"  Test statistic W = {stat:.2f}")
print(f"  p-value = {pval:.4f}")
print(f"  Statistically significant (p < 0.05): {pval < 0.05}")


## 9. Model Performance Visualizations
We generate publication-ready diagnostic plots:
1. **Model Ladder Comparison:** R² scores across model families with B1 baseline and B2 target lines.
2. **True vs. Predicted Diagnostics:** Parity scatter plots comparing idealized and realistic geometries.
3. **Feature Importance:** Top drivers of pedestrian-level wind speed.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model ladder
ax = axes[0]
mnames = list(ladder_results.keys())
means = [np.mean(ladder_results[n]["r2s"]) for n in mnames]
stds  = [np.std(ladder_results[n]["r2s"]) for n in mnames]
ax.bar(mnames, means, yerr=stds, capsize=5, alpha=0.85, color="#81a1c1", edgecolor="#2e3440")
ax.axhline(b1_mean, color="#ebcb8b", linestyle="--", linewidth=1.5, label=f"B1 Baseline ({b1_mean:.3f})")
ax.axhline(0.867, color="#a3be8c", linestyle="--", linewidth=1.5, label="B2 Target (0.867)")
ax.set_ylabel("Grouped 5-Fold CV R²")
ax.set_title("Model Ladder Performance (U_mean)")
ax.set_ylim(0, 1.0)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)

# Parity plot
ax = axes[1]
ideal_mask = (full_df["subset"] == "idealized").values
real_mask  = (full_df["subset"] == "realistic").values
lgbm_oof = ladder_results["LightGBM"]["oof"]

ax.scatter(y_umean[ideal_mask], lgbm_oof[ideal_mask], alpha=0.5, label="Idealized", color="#81a1c1")
ax.scatter(y_umean[real_mask], lgbm_oof[real_mask], alpha=0.5, label="Realistic", color="#bf616a")
lo = min(y_umean.min(), lgbm_oof.min()) - 0.05
hi = max(y_umean.max(), lgbm_oof.max()) + 0.05
ax.plot([lo, hi], [lo, hi], "k--", alpha=0.7)
ax.set_xlabel("True U_mean (U / u*)")
ax.set_ylabel("Predicted U_mean (U / u*)")
ax.set_title("True vs Predicted Parity Plot (OOF)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


## 10. Stage Gate Verification & Contract Predictions
We verify gate passage criteria:
- Grouped CV $R^2(U_{\text{mean}}) > 0.671$ (B1)
- Grouped CV $R^2(U_{p95}) > 0.664$ (B1)
- Wilcoxon test $p < 0.05$


In [ ]:
B1_UMEAN = 0.671
B1_UP95  = 0.664

umean_r2 = float(np.mean(ladder_results["LightGBM"]["r2s"]))
up95_r2  = float(np.mean(lgbm_results["U_p95"]["r2s"]))
gate_passed = (umean_r2 > B1_UMEAN) and (up95_r2 > B1_UP95) and (pval < 0.05)

gate_summary = pd.DataFrame([
    {"Criterion": "U_mean CV R² > B1 (0.671)", "Value": f"{umean_r2:.4f}", "Status": "PASS" if umean_r2 > B1_UMEAN else "FAIL"},
    {"Criterion": "U_p95 CV R² > B1 (0.664)", "Value": f"{up95_r2:.4f}", "Status": "PASS" if up95_r2 > B1_UP95 else "FAIL"},
    {"Criterion": "Wilcoxon Signed-Rank p < 0.05", "Value": f"{pval:.4f}", "Status": "PASS" if pval < 0.05 else "FAIL"},
])

display(gate_summary)
print(f"OVERALL STAGE S06 GATE STATUS: {'PASSED ✅' if gate_passed else 'FAILED ❌'}")
